In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("sample") \
    .getOrCreate()

# Jupyter コンテナ内のパスで CSV を読み込み
orders = spark.read.csv("/home/jovyan/fixtures/data/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("/home/jovyan/fixtures/data/customers.csv", header=True, inferSchema=True)

print("=== orders ===")
orders.show()
print("=== customers ===")
customers.show()

=== orders ===
+--------+-----------+------+---------+------+----------+
|order_id|customer_id|region|   status|amount|order_date|
+--------+-----------+------+---------+------+----------+
|       1|       C001|  East|   active| 150.0|2024-01-10|
|       2|       C001|  East|   active| 200.0|2024-01-15|
|       3|       C002|  West|   active| 300.0|2024-01-20|
|       4|       C002|  West|cancelled| 100.0|2024-01-22|
|       5|       C003|  East|   active| 250.0|2024-02-01|
|       6|       C003|  East|   active| 175.0|2024-02-05|
|       7|       C004|  West|   active| 400.0|2024-02-10|
|       8|       C004| North|   active| 120.0|2024-02-12|
|       9|       C005| North|   active|  90.0|2024-02-20|
|      10|       C005| North|cancelled|  60.0|2024-02-25|
+--------+-----------+------+---------+------+----------+

=== customers ===
+-----------+------------+-----------------+------+
|customer_id|        name|            email|  tier|
+-----------+------------+-----------------+------

In [ ]:
# フィルタ: active のみ
active_orders = orders.filter(F.col("status") == "active")

# JOIN: 顧客情報を結合
joined = active_orders.join(customers, on=["customer_id"], how="left")

print("=== active orders + customers ===")
joined.show()

In [ ]:
# 集計: 顧客 x リージョンごとの合計・件数・最終注文日
summary = (
    joined
    .groupBy("customer_id", "region")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("order_id").alias("order_count"),
        F.max("order_date").alias("last_order_date"),
    )
)

print("=== summary ===")
summary.show()

In [ ]:
# Window 関数: リージョン内で売上順にランキング
window_spec = Window.partitionBy("region").orderBy(F.col("total_amount").desc())
ranked = summary.withColumn("rank", F.row_number().over(window_spec))

# 上位10件を抽出して表示
result = (
    ranked
    .filter(F.col("rank") <= 10)
    .select("customer_id", "region", "total_amount", "order_count", "last_order_date", "rank")
    .orderBy("region", "rank")
)

print("=== result (ranked by region) ===")
result.show()